In [7]:
def fifo(reference, frame_size):
    frames = [None] * frame_size
    oldest = 0
    hits = faults = 0
    insert_seq, replace_seq, rows = [], [], []

    for page in reference:
        if page in frames:
            hits += 1
            status, action = "HIT", "-"
        else:
            faults += 1
            status = "MISS (FAULT)"
            if None in frames:
                frames[frames.index(None)] = page
                insert_seq.append(page)
                action = f"Insert {page}"
            else:
                victim = frames[oldest]
                frames[oldest] = page
                replace_seq.append((victim, page))
                oldest = (oldest + 1) % frame_size
                action = f"Replace {victim} -> {page}"
        rows.append((page, list(frames), status, action))

    total = len(reference)
    return dict(rows=rows, hits=hits, faults=faults,
                hit_ratio=hits/total, fault_ratio=faults/total,
                insert_seq=insert_seq, replace_seq=replace_seq, total=total)


def print_result(reference, frame_size):
    r = fifo(reference, frame_size)
    print(f"Reference String : {' '.join(map(str, reference))}")
    print(f"Frame Size       : {frame_size}\n")

    head = f"{'Page':<6}" + "".join(f"{'F'+str(i+1):<5}" for i in range(frame_size)) \
           + f"{'Status':<15}Action"
    print(head)
    print("-" * (len(head) + 12))
    for page, fr, status, action in r["rows"]:
        cells = "".join(f"{('-' if x is None else x)!s:<5}" for x in fr)
        print(f"{page:<6}{cells}{status:<15}{action}")

    print()
    print(f"Total References : {r['total']}")
    print(f"Hits             : {r['hits']}")
    print(f"Misses (Faults)  : {r['faults']}")
    print(f"Hit Ratio        : {r['hits']}/{r['total']} = {r['hit_ratio']:.4f} ({r['hit_ratio']*100:.2f}%)")
    print(f"Fault Ratio      : {r['faults']}/{r['total']} = {r['fault_ratio']:.4f} ({r['fault_ratio']*100:.2f}%)")
    print(f"Insert Sequence  : {' -> '.join(map(str, r['insert_seq']))}")
    rep = ", ".join(f"{v}->{n}" for v, n in r["replace_seq"]) or "None"
    print(f"Replace Sequence : {rep}   (old page -> new page)")

reference = [1, 2, 3, 4, 5, 1, 2, 3, 1]
frame_size = 3
print_result(reference, frame_size)


Reference String : 1 2 3 4 5 1 2 3 1
Frame Size       : 3

Page  F1   F2   F3   Status         Action
------------------------------------------------------
1     1    -    -    MISS (FAULT)   Insert 1
2     1    2    -    MISS (FAULT)   Insert 2
3     1    2    3    MISS (FAULT)   Insert 3
4     4    2    3    MISS (FAULT)   Replace 1 -> 4
5     4    5    3    MISS (FAULT)   Replace 2 -> 5
1     4    5    1    MISS (FAULT)   Replace 3 -> 1
2     2    5    1    MISS (FAULT)   Replace 4 -> 2
3     2    3    1    MISS (FAULT)   Replace 5 -> 3
1     2    3    1    HIT            -

Total References : 9
Hits             : 1
Misses (Faults)  : 8
Hit Ratio        : 1/9 = 0.1111 (11.11%)
Fault Ratio      : 8/9 = 0.8889 (88.89%)
Insert Sequence  : 1 -> 2 -> 3
Replace Sequence : 1->4, 2->5, 3->1, 4->2, 5->3   (old page -> new page)
